# Notebook 02 — Preprocesamiento

**Responsable:** Estudiante A  
**Objetivo:** Transformar el dataset raw en los artefactos necesarios para los modelos.

## Contenido
1. Parsing de columnas JSON
2. Construcción del "tag soup" (para Content-Based)
3. Limpieza de datos numéricos
4. Simulación de la matriz de ratings (para Collaborative Filtering)
5. División train/test
6. Guardado de artefactos procesados

In [1]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import sys

sys.path.insert(0, os.path.abspath('..'))

from src.preprocessing.json_parser import extract_all_features
from src.preprocessing.feature_engineer import add_tag_soup, normalize_numeric
from src.preprocessing.train_test_split import split_ratings

## 1. Carga y Join del Dataset

In [2]:
movies  = pd.read_csv('../data/raw/tmdb_5000_movies.csv')
credits = pd.read_csv('../data/raw/tmdb_5000_credits.csv')

credits = credits.rename(columns={'movie_id': 'id'})
df = movies.merge(credits[['id', 'cast', 'crew']], on='id', how='left')

print(f'Shape despues del merge: {df.shape}')

null_counts = df.isnull().sum()
null_cols = null_counts[null_counts > 0]
print('Columnas con nulos:')
print(null_cols.to_string())

CRITICAL = ['title', 'genres', 'vote_average', 'vote_count']
criticos_nulos = df[CRITICAL].isnull().sum().sum()
assert criticos_nulos == 0, f'Nulos en columnas criticas: {df[CRITICAL].isnull().sum().to_dict()}'
print('Verificacion de nulos en columnas criticas: OK')


Shape despues del merge: (4803, 22)
Columnas con nulos:
homepage        3091
overview           3
release_date       1
runtime            2
tagline          844
Verificacion de nulos en columnas criticas: OK


## 2. Parsing de Columnas JSON

Las columnas `genres`, `keywords`, `cast` y `crew` contienen strings con formato Python dict (comillas simples). Usamos `ast.literal_eval()` para convertirlas en listas de Python.

In [3]:
# Aplicar parseo de todas las columnas JSON
df = extract_all_features(df)

# Verificar los resultados
print('Ejemplo genres_list:  ', df['genres_list'].iloc[0])
print('Ejemplo keywords_list:', df['keywords_list'].iloc[0][:5])
print('Ejemplo cast_list:    ', df['cast_list'].iloc[0])
print('Ejemplo director:     ', df['director'].iloc[0])

Ejemplo genres_list:   ['Action', 'Adventure', 'Fantasy', 'Science Fiction']
Ejemplo keywords_list: ['culture clash', 'future', 'space war', 'space colony', 'society']
Ejemplo cast_list:     ['Sam Worthington', 'Zoe Saldana', 'Sigourney Weaver']
Ejemplo director:      James Cameron


## 3. Construcción del Tag Soup

Combina géneros (x2), keywords, actores y director (x2) en una cadena por película.
Los espacios dentro de nombres se eliminan para que sean tokens únicos.

Ejemplo: `"action adventurepax iceplanet samworthington zoesaldana jamescameron jamescameron"`

In [4]:
df = add_tag_soup(df)

# Mostrar ejemplos
print('Tag soup de las primeras 3 películas:\n')
for _, row in df[['title', 'tag_soup']].head(3).iterrows():
    print(f"  {row['title']}:")
    print(f"  {row['tag_soup'][:120]}...\n")

Tag soup de las primeras 3 películas:

  Avatar:
  action action adventure adventure fantasy fantasy sciencefiction sciencefiction cultureclash future spacewar spacecolony...

  Pirates of the Caribbean: At World's End:
  adventure adventure fantasy fantasy action action ocean drugabuse exoticisland eastindiatradingcompany loveofone'slife t...

  Spectre:
  action action adventure adventure crime crime spy basedonnovel secretagent sequel mi6 britishsecretservice unitedkingdom...



## 4. Limpieza de Variables Numéricas

In [5]:
# Normalización log1p de budget y revenue (distribuciones muy sesgadas)
df = normalize_numeric(df, ['budget', 'revenue'])

# Extraer año de release_date
df['release_year'] = pd.to_datetime(df['release_date'], errors='coerce').dt.year

# Filtrar películas con vote_count >= 50 para Collaborative Filtering
# (evitar ruido de películas con muy pocos votos)
df_cf = df[df['vote_count'] >= 50].copy().reset_index(drop=True)
print(f'Películas con vote_count >= 50: {len(df_cf)} (de {len(df)} totales)')

Películas con vote_count >= 50: 3652 (de 4803 totales)


## 5. Simulación de la Matriz de Ratings

El dataset TMDB **no tiene ratings por usuario**. Solo tiene `vote_average` y `vote_count`.

**Estrategia:** Simular N usuarios sintéticos cuyos ratings siguen una distribución
Normal(vote_average, 1). Esto nos permite construir una matriz usuario-película
para los algoritmos de Collaborative Filtering.

**Limitación:** Los usuarios son sintéticos — documentar esto en el informe.

Para mantener la simulación manejable, limitamos a 500 usuarios y 200 películas.

In [6]:
rng = np.random.default_rng(42)

N_USERS  = 500   # usuarios sintéticos
N_MOVIES = min(200, len(df_cf))  # películas (subconjunto manejable)
DENSITY  = 0.15  # fracción de ratings conocidos por usuario (~15%)

# Seleccionar N_MOVIES películas
df_sample = df_cf.head(N_MOVIES).copy()
vote_avgs = df_sample['vote_average'].values  # shape: (N_MOVIES,)

# Crear matriz de ratings (0 = desconocido)
ratings_matrix = np.zeros((N_USERS, N_MOVIES))

for u in range(N_USERS):
    # Elegir aleatoriamente qué películas calificó este usuario
    n_rated = max(5, int(N_MOVIES * DENSITY))
    rated_movies = rng.choice(N_MOVIES, size=n_rated, replace=False)

    for m in rated_movies:
        # Rating = Normal(vote_average, 1), recortado a [1, 10]
        rating = rng.normal(vote_avgs[m], 1.0)
        ratings_matrix[u, m] = float(np.clip(rating, 1.0, 10.0))

sparsity = (ratings_matrix == 0).mean()
print(f'Matriz de ratings: {ratings_matrix.shape}')
print(f'Densidad: {1 - sparsity:.1%} | Sparsity: {sparsity:.1%}')
print(f'Rango de ratings: [{ratings_matrix[ratings_matrix > 0].min():.1f}, {ratings_matrix[ratings_matrix > 0].max():.1f}]')

Matriz de ratings: (500, 200)
Densidad: 15.0% | Sparsity: 85.0%
Rango de ratings: [1.0, 10.0]


## 6. División Train / Test

In [7]:
train_matrix, test_matrix = split_ratings(ratings_matrix, test_ratio=0.2, seed=42)

print(f'Ratings en train: {(train_matrix > 0).sum()}')
print(f'Ratings en test:  {(test_matrix > 0).sum()}')

Ratings en train: 12000


Ratings en test:  3000


## 7. Guardado de Artefactos

In [8]:
os.makedirs('../data/processed', exist_ok=True)

# Dataset limpio (para content-based)
df[['id', 'title', 'tag_soup', 'genres_list', 'vote_average', 'vote_count',
    'budget_log', 'revenue_log', 'release_year']].to_csv(
    '../data/processed/movies_clean.csv', index=False)

# Subconjunto de películas para CF
df_sample[['id', 'title', 'vote_average']].to_csv(
    '../data/processed/movies_cf.csv', index=False)

# Matrices de ratings
np.save('../data/processed/ratings_train.npy', train_matrix)
np.save('../data/processed/ratings_test.npy',  test_matrix)

print('Artefactos guardados en data/processed/:')
for f in os.listdir('../data/processed'):
    size = os.path.getsize(f'../data/processed/{f}') / 1024
    print(f'  {f} ({size:.1f} KB)')

Artefactos guardados en data/processed/:
  .gitkeep (0.0 KB)
  movies_cf.csv (6.0 KB)
  movies_clean.csv (1321.0 KB)
  ratings_test.npy (781.4 KB)
  ratings_train.npy (781.4 KB)
